In [1]:
from constants import ORACLE_SKILLS, SKIP_DIRS
from pathlib import Path
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

headers = [ ("#", "h1"), ("##", "h2"), ("###", "h3") ]

section_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers, strip_headers = True)

child_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name = "cl100k_base",
    chunk_size = 500,
    chunk_overlap = 40,
    separators = ["\n\n", "\n", ". ", " ", ""],
)

def build_chunks(md_text: str, source_path: str):
  sections = section_splitter.split_text(md_text)
  chunks = []

  for section_id, doc in enumerate(sections):
    h1 = doc.metadata.get("h1")
    h2 = doc.metadata.get("h2")
    h3 = doc.metadata.get("h3")

    path_parts = [x for x in [h1, h2, h3] if x]
    # header_path = " > ".join(path_parts)

    body = doc.page_content.strip()

    # Text used for embedding
    # embedding_text = f"{header_path}\n\n{body}" if header_path else body
    # embedding_text = body

    for chunk_id, piece in enumerate(child_splitter.split_text(body)):
      # chunks.append({ "source": source_path, "section_id": section_id, "chunk_id": chunk_id, "h1": h1, "h2": h2, "h3": h3, "header_path": path_parts, "text": piece })
      chunks.append({ "source": source_path, "section_id": section_id, "chunk_id": chunk_id, "header_path": path_parts, "chunk": piece })

  return chunks

docs_root = Path(ORACLE_SKILLS)
markdown_files = []
for path in sorted(docs_root.rglob("*.md")):
  if any(part in SKIP_DIRS for part in path.parts):
    continue
  if path.is_file():
    markdown_files.append(path)
# print(f"Found {len(markdown_files)} markdown files in {docs_root}.")
chunks = []
i = 0
for path in markdown_files:
  # chunks.append({ "file": str(path), "chunks": chunk_by_sections(path.read_text(encoding="utf-8", errors="ignore").strip().lower()) })
  chunk_sections = build_chunks(md_text = path.read_text(encoding = "utf-8", errors = "ignore").strip().lower(), source_path = str(path))
  for chunk in chunk_sections:
    # chunks.append({"id": i, "source": chunk['source'], "section_id": chunk['section_id'], "chunk_id": chunk['chunk_id'], "header_path": chunk['header_path'], "chunk": chunk['text']})
    chunks.append({"id": i, **chunk})
    # chunks.append({"id": i, "file": chunk['metadata']['source'], "chunk": chunk['text']})
    i += 1

c:\Users\yjaquier\.conda\envs\ollama\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
for item in list(filter(lambda c: "awr-report" in c.get("source", ""), chunks))[:10]:
  print(item)

{'id': 2536, 'source': 'skills-main\\db\\performance\\awr-reports.md', 'section_id': 0, 'chunk_id': 0, 'header_path': ['awr reports — automatic workload repository', 'overview'], 'chunk': "the automatic workload repository (awr) is oracle's built-in performance data collection and analysis framework. it automatically captures snapshots of performance statistics at regular intervals (default: every 60 minutes) and stores them in the sysaux tablespace. awr data is the foundation for diagnosing performance problems, understanding workload trends, and validating the impact of tuning changes.  \nawr reports compare two snapshots and summarize the activity between them. they are the first tool most dbas reach for when investigating a performance incident.  \n**licensing note:** awr is part of the oracle diagnostics pack. it requires a license in addition to the base database license. verify your license before using awr in production.  \n---"}
{'id': 2537, 'source': 'skills-main\\db\\perform